# Calculate Scattering Intensity $I(Q)$ — mAb1 174-Bead Model

This notebook runs Monte Carlo (MC) simulations of a monoclonal antibody (mAb1) using a coarse-grained 174-bead representation, and plots the resulting scattering intensity $I(Q)$ against experimental SAXS data.

**Workflow:**
1. Define simulation parameters (pH, ionic strength, concentration, etc.)
2. Generate Faunus input files and run MC simulations (Yukawa + Lennard-Jones potentials)
3. Load experimental SAXS data
4. Plot and compare simulated vs. experimental $I(Q)$

**Dependencies:** `numpy`, `pandas`, `scipy`, `matplotlib`, `jinja2`, [Faunus](https://github.com/mlund/faunus), `yason.py`

**Required input files (relative to `workdir`):**
- `Inputs/mAb_net_charge.cvs` — net charge table (columns: `pH`, `IonicStrength`, `NetCharge`)
- `Inputs/mAb1_174-bead_{i}_mM_pH_{p}.pqr` — per-condition PQR structure files
- `Experimental_data/45_10_mAb1_saxs.dat`, `45_22p5_mAb1_saxs.dat`, `45_150_mAb1_saxs.dat` — SAXS profiles

## 1. Import Packages

In [ ]:
import numpy as np
import pandas as pd
from jinja2 import Template
import os
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import interpolate
from IPython.display import display, HTML

# Widen the notebook display
display(HTML("""
<style>
    div#notebook-container    { width: 100%; }
    div#menubar-container     { width: 100%; }
    div#maintoolbar-container { width: 100%; }
</style>
"""))

## 2. Set Working Directory

The working directory is captured on first run and restored on subsequent executions, so the notebook can be re-run safely from any cell.

In [ ]:
try:
    workdir
except NameError:
    workdir = %pwd
else:
    %cd -q $workdir

print(f"Working directory: {workdir}")

## 3. Helper Functions

In [ ]:
def read_charge(path_Q, pH, i, tol=1e-8):
    """Read net charge from a CSV table for a given pH and ionic strength.

    Parameters
    ----------
    path_Q : str
        Path to the CSV file with columns: pH, IonicStrength, NetCharge.
    pH : float
        Target pH value.
    i : float
        Target ionic strength (Molar).
    tol : float, optional
        Numerical tolerance for matching float values (default 1e-8).

    Returns
    -------
    float
        Net charge (in units of elementary charge).
    """
    df = pd.read_csv(path_Q)
    df.columns = [c.strip() for c in df.columns]  # strip whitespace from headers

    row = df[
        (df["pH"].astype(float).sub(float(pH)).abs() <= tol) &
        (df["IonicStrength"].astype(float).sub(float(i)).abs() <= tol)
    ]

    if row.empty:
        raise ValueError(f"No charge found for pH={pH}, I={i} in {path_Q}")

    return float(row.iloc[0]["NetCharge"])


def length_in_gl(cp, N_of_proteins, Mw):
    """Compute the cubic box side length for a given protein concentration.

    Parameters
    ----------
    cp : float
        Protein concentration in g/L.
    N_of_proteins : int
        Number of protein molecules in the simulation box.
    Mw : float
        Molecular weight (g/mol).

    Returns
    -------
    float
        Side length of the simulation cube in Angstroms.
    """
    return np.cbrt(N_of_proteins / (cp * (1 / Mw) * 6.022e23 * 1e-27))


def debye_length(Mw, Cp, Z, rhos, rhob):
    """Compute the Debye screening length.

    Parameters
    ----------
    Mw : float
        Molecular weight (g/mol).
    Cp : float
        Protein concentration (mg/mL).
    Z : float
        Net charge of protein (elementary charges).
    rhos : float
        Salt concentration (Molar).
    rhob : float
        Buffer ionic strength (Molar).

    Returns
    -------
    float
        Debye length in Angstroms.
    """
    lB = 7.14      # Bjerrum length [Å]
    Na = 6.02214076e23
    k = np.sqrt(
        4 * np.pi * lB * (
            Cp * Z * Na / Mw * 1e-27
            + 2 * rhos * Na * 1e-27
            + 2 * rhob * Na * 1e-27
        )
    )
    return 1.0 / k


def create_input(path_monomer, macro, micro, nstep, nskip, L, n, Dl, e, eps, i):
    """Write the Faunus YAML input file for a single simulation run.

    Parameters
    ----------
    path_monomer : str
        Path to the PQR structure file.
    macro : int
        Number of macro (outer) MC steps.
    micro : int
        Number of micro (inner) MC steps.
    nstep : int
        Analysis/output frequency (steps).
    nskip : int
        Equilibration steps to skip before analysis.
    L : float
        Cubic box side length [Å].
    n : int
        Number of mAb molecules.
    Dl : float
        Debye length [Å].
    e : int
        Epsilon index (not directly used in template, kept for labelling).
    eps : float
        Lennard-Jones epsilon [kJ/mol].
    i : float
        Ionic strength [Molar].

    Returns
    -------
    int
        Number of characters written to the input file.
    """
    with open('input.yml', 'w') as input_file:
        return input_file.write(
            Input.render(
                path_monomer=path_monomer,
                macro=macro,
                micro=micro,
                L=L,
                n=n,
                nstep=nstep,
                nskip=nskip,
                Dl=Dl,
                e=e,
                eps=eps,
                i=i,
            )
        )

## 4. Faunus Input Template

The YAML template below is rendered by Jinja2 for each simulation condition. It specifies:
- **Geometry**: cubic box of side length `L`
- **Energy**: non-bonded Coulomb (Yukawa) + Lennard-Jones (Lorentz-Berthelot mixing)
- **Atomlist**: 174 coarse-grained beads representing mAb1
- **Analysis**: energy, density, trajectory (XTC), and Debye scattering

In [ ]:
Input = Template("""
temperature: 298.15
geometry: {type: cuboid, length: [{{L}},{{L}},{{L}}]}
mcloop: {micro: {{micro}}, macro: {{macro}}}
random: {seed: hardware}
energy:
    - nonbonded_coulomblj:
        #summation_policy: openmp
        #cutoff_g2g: {{ 92 + 8*Dl }} # 92 Å ≈ mAb diameter; cutoff = diameter + 8·λD
        lennardjones: { mixing: LB }
        coulomb: { type: yukawa, debyelength: {{Dl}}, epsr: 78.7, shift: True, cutoff: {{8*Dl}} }

atomlist:
  - B1:   {mw: 840.52, sigma: 11.71, eps: {{eps}}, q: 0 }
  - B2:   {mw: 652.36, sigma: 10.76, eps: {{eps}}, q: 0 }
  - B3:   {mw: 876.56, sigma: 11.87, eps: {{eps}}, q: 0 }
  - B4:   {mw: 792.46, sigma: 11.48, eps: {{eps}}, q: 0 }
  - B5:   {mw: 944.61, sigma: 12.17, eps: {{eps}}, q: 0 }
  - B6:   {mw: 784.53, sigma: 11.44, eps: {{eps}}, q: 0 }
  - B7:   {mw: 902.55, sigma: 11.99, eps: {{eps}}, q: 0 }
  - B8:   {mw: 738.43, sigma: 11.21, eps: {{eps}}, q: 0 }
  - B9:   {mw: 736.39, sigma: 11.20, eps: {{eps}}, q: 0 }
  - B10:  {mw: 774.46, sigma: 11.39, eps: {{eps}}, q: 0 }
  - B11:  {mw: 924.60, sigma: 12.09, eps: {{eps}}, q: 0 }
  - B12:  {mw: 942.57, sigma: 12.16, eps: {{eps}}, q: 0 }
  - B13:  {mw: 760.46, sigma: 11.32, eps: {{eps}}, q: 0 }
  - B14:  {mw: 800.48, sigma: 11.52, eps: {{eps}}, q: 0 }
  - B15:  {mw: 820.56, sigma: 11.62, eps: {{eps}}, q: 0 }
  - B16:  {mw: 788.43, sigma: 11.46, eps: {{eps}}, q: 0 }
  - B17:  {mw: 724.49, sigma: 11.14, eps: {{eps}}, q: 0 }
  - B18:  {mw: 930.57, sigma: 12.11, eps: {{eps}}, q: 0 }
  - B19:  {mw: 884.56, sigma: 11.91, eps: {{eps}}, q: 0 }
  - B20:  {mw: 734.40, sigma: 11.19, eps: {{eps}}, q: 0 }
  - B21:  {mw: 822.42, sigma: 11.62, eps: {{eps}}, q: 0 }
  - B22:  {mw: 822.46, sigma: 11.62, eps: {{eps}}, q: 0 }
  - B23:  {mw: 738.43, sigma: 11.21, eps: {{eps}}, q: 0 }
  - B24:  {mw: 992.61, sigma: 12.38, eps: {{eps}}, q: 0 }
  - B25:  {mw: 774.49, sigma: 11.39, eps: {{eps}}, q: 0 }
  - B26:  {mw: 738.43, sigma: 11.21, eps: {{eps}}, q: 0 }
  - B27:  {mw: 892.56, sigma: 11.95, eps: {{eps}}, q: 0 }
  - B28:  {mw: 678.38, sigma: 10.90, eps: {{eps}}, q: 0 }
  - B29:  {mw: 784.48, sigma: 11.44, eps: {{eps}}, q: 0 }
  - B30:  {mw: 694.45, sigma: 10.99, eps: {{eps}}, q: 0 }
  - B31:  {mw: 952.65, sigma: 12.21, eps: {{eps}}, q: 0 }
  - B32:  {mw: 894.58, sigma: 11.95, eps: {{eps}}, q: 0 }
  - B33:  {mw: 796.49, sigma: 11.50, eps: {{eps}}, q: 0 }
  - B34:  {mw: 824.51, sigma: 11.63, eps: {{eps}}, q: 0 }
  - B35:  {mw: 756.47, sigma: 11.30, eps: {{eps}}, q: 0 }
  - B36:  {mw: 922.55, sigma: 12.08, eps: {{eps}}, q: 0 }
  - B37:  {mw: 850.52, sigma: 11.76, eps: {{eps}}, q: 0 }
  - B38:  {mw: 856.53, sigma: 11.78, eps: {{eps}}, q: 0 }
  - B39:  {mw: 874.60, sigma: 11.87, eps: {{eps}}, q: 0 }
  - B40:  {mw: 800.48, sigma: 11.52, eps: {{eps}}, q: 0 }
  - B41:  {mw: 854.51, sigma: 11.77, eps: {{eps}}, q: 0 }
  - B42:  {mw: 688.39, sigma: 10.95, eps: {{eps}}, q: 0 }
  - B43:  {mw: 674.39, sigma: 10.88, eps: {{eps}}, q: 0 }
  - B44:  {mw: 810.57, sigma: 11.57, eps: {{eps}}, q: 0 }
  - B45:  {mw: 688.35, sigma: 10.95, eps: {{eps}}, q: 0 }
  - B46:  {mw: 826.56, sigma: 11.64, eps: {{eps}}, q: 0 }
  - B47:  {mw: 796.49, sigma: 11.50, eps: {{eps}}, q: 0 }
  - B48:  {mw: 764.45, sigma: 11.34, eps: {{eps}}, q: 0 }
  - B49:  {mw: 752.48, sigma: 11.28, eps: {{eps}}, q: 0 }
  - B50:  {mw: 778.45, sigma: 11.41, eps: {{eps}}, q: 0 }
  - B51:  {mw: 720.44, sigma: 11.12, eps: {{eps}}, q: 0 }
  - B52:  {mw: 706.39, sigma: 11.05, eps: {{eps}}, q: 0 }
  - B53:  {mw: 900.58, sigma: 11.98, eps: {{eps}}, q: 0 }
  - B54:  {mw: 806.47, sigma: 11.55, eps: {{eps}}, q: 0 }
  - B55:  {mw: 852.52, sigma: 11.76, eps: {{eps}}, q: 0 }
  - B56:  {mw: 712.55, sigma: 11.08, eps: {{eps}}, q: 0 }
  - B57:  {mw: 604.36, sigma: 10.49, eps: {{eps}}, q: 0 }
  - B58:  {mw: 830.57, sigma: 11.66, eps: {{eps}}, q: 0 }
  - B59:  {mw: 680.37, sigma: 10.91, eps: {{eps}}, q: 0 }
  - B60:  {mw: 764.51, sigma: 11.34, eps: {{eps}}, q: 0 }
  - B61:  {mw: 736.39, sigma: 11.20, eps: {{eps}}, q: 0 }
  - B62:  {mw: 986.63, sigma: 12.35, eps: {{eps}}, q: 0 }
  - B63:  {mw: 726.42, sigma: 11.15, eps: {{eps}}, q: 0 }
  - B64:  {mw: 868.51, sigma: 11.84, eps: {{eps}}, q: 0 }
  - B65:  {mw: 844.49, sigma: 11.73, eps: {{eps}}, q: 0 }
  - B66:  {mw: 784.48, sigma: 11.44, eps: {{eps}}, q: 0 }
  - B67:  {mw: 948.60, sigma: 12.19, eps: {{eps}}, q: 0 }
  - B68:  {mw: 666.43, sigma: 10.84, eps: {{eps}}, q: 0 }
  - B69:  {mw: 724.42, sigma: 11.14, eps: {{eps}}, q: 0 }
  - B70:  {mw: 684.41, sigma: 10.93, eps: {{eps}}, q: 0 }
  - B71:  {mw: 862.53, sigma: 11.81, eps: {{eps}}, q: 0 }
  - B72:  {mw: 868.53, sigma: 11.84, eps: {{eps}}, q: 0 }
  - B73:  {mw: 796.51, sigma: 11.50, eps: {{eps}}, q: 0 }
  - B74:  {mw: 834.52, sigma: 11.68, eps: {{eps}}, q: 0 }
  - B75:  {mw: 868.51, sigma: 11.84, eps: {{eps}}, q: 0 }
  - B76:  {mw: 872.50, sigma: 11.86, eps: {{eps}}, q: 0 }
  - B77:  {mw: 752.43, sigma: 11.28, eps: {{eps}}, q: 0 }
  - B78:  {mw: 860.53, sigma: 11.80, eps: {{eps}}, q: 0 }
  - B79:  {mw: 882.53, sigma: 11.90, eps: {{eps}}, q: 0 }
  - B80:  {mw: 888.55, sigma: 11.93, eps: {{eps}}, q: 0 }
  - B81:  {mw: 812.58, sigma: 11.58, eps: {{eps}}, q: 0 }
  - B82:  {mw: 934.56, sigma: 12.13, eps: {{eps}}, q: 0 }
  - B83:  {mw: 784.48, sigma: 11.44, eps: {{eps}}, q: 0 }
  - B84:  {mw: 718.40, sigma: 11.11, eps: {{eps}}, q: 0 }
  - B85:  {mw: 754.43, sigma: 11.29, eps: {{eps}}, q: 0 }
  - B86:  {mw: 776.48, sigma: 11.40, eps: {{eps}}, q: 0 }
  - B87:  {mw: 834.52, sigma: 11.68, eps: {{eps}}, q: 0 }
  - B88:  {mw: 840.50, sigma: 11.71, eps: {{eps}}, q: 0 }
  - B89:  {mw: 868.56, sigma: 11.84, eps: {{eps}}, q: 0 }
  - B90:  {mw: 792.47, sigma: 11.48, eps: {{eps}}, q: 0 }
  - B91:  {mw: 746.41, sigma: 11.25, eps: {{eps}}, q: 0 }
  - B92:  {mw: 788.47, sigma: 11.46, eps: {{eps}}, q: 0 }
  - B93:  {mw: 910.53, sigma: 12.03, eps: {{eps}}, q: 0 }
  - B94:  {mw: 940.64, sigma: 12.16, eps: {{eps}}, q: 0 }
  - B95:  {mw: 838.51, sigma: 11.70, eps: {{eps}}, q: 0 }
  - B96:  {mw: 764.45, sigma: 11.34, eps: {{eps}}, q: 0 }
  - B97:  {mw: 748.45, sigma: 11.26, eps: {{eps}}, q: 0 }
  - B98:  {mw: 840.54, sigma: 11.71, eps: {{eps}}, q: 0 }
  - B99:  {mw: 756.43, sigma: 11.30, eps: {{eps}}, q: 0 }
  - B100: {mw: 780.51, sigma: 11.42, eps: {{eps}}, q: 0 }
  - B101: {mw: 920.59, sigma: 12.07, eps: {{eps}}, q: 0 }
  - B102: {mw: 888.55, sigma: 11.93, eps: {{eps}}, q: 0 }
  - B103: {mw: 770.39, sigma: 11.37, eps: {{eps}}, q: 0 }
  - B104: {mw: 844.45, sigma: 11.73, eps: {{eps}}, q: 0 }
  - B105: {mw: 772.42, sigma: 11.38, eps: {{eps}}, q: 0 }
  - B106: {mw: 826.50, sigma: 11.64, eps: {{eps}}, q: 0 }
  - B107: {mw: 892.60, sigma: 11.95, eps: {{eps}}, q: 0 }
  - B108: {mw: 794.45, sigma: 11.49, eps: {{eps}}, q: 0 }
  - B109: {mw: 800.48, sigma: 11.52, eps: {{eps}}, q: 0 }
  - B110: {mw: 868.54, sigma: 11.84, eps: {{eps}}, q: 0 }
  - B111: {mw: 740.43, sigma: 11.22, eps: {{eps}}, q: 0 }
  - B112: {mw: 812.49, sigma: 11.58, eps: {{eps}}, q: 0 }
  - B113: {mw: 722.45, sigma: 11.13, eps: {{eps}}, q: 0 }
  - B114: {mw: 940.62, sigma: 12.16, eps: {{eps}}, q: 0 }
  - B115: {mw: 790.54, sigma: 11.47, eps: {{eps}}, q: 0 }
  - B116: {mw: 910.66, sigma: 12.03, eps: {{eps}}, q: 0 }
  - B117: {mw: 832.48, sigma: 11.67, eps: {{eps}}, q: 0 }
  - B118: {mw: 786.48, sigma: 11.45, eps: {{eps}}, q: 0 }
  - B119: {mw: 754.43, sigma: 11.29, eps: {{eps}}, q: 0 }
  - B120: {mw: 876.53, sigma: 11.87, eps: {{eps}}, q: 0 }
  - B121: {mw: 718.40, sigma: 11.11, eps: {{eps}}, q: 0 }
  - B122: {mw: 970.72, sigma: 12.28, eps: {{eps}}, q: 0 }
  - B123: {mw: 990.57, sigma: 12.37, eps: {{eps}}, q: 0 }
  - B124: {mw: 780.49, sigma: 11.42, eps: {{eps}}, q: 0 }
  - B125: {mw: 706.39, sigma: 11.05, eps: {{eps}}, q: 0 }
  - B126: {mw: 712.46, sigma: 11.08, eps: {{eps}}, q: 0 }
  - B127: {mw: 794.48, sigma: 11.49, eps: {{eps}}, q: 0 }
  - B128: {mw: 664.44, sigma: 10.83, eps: {{eps}}, q: 0 }
  - B129: {mw: 910.57, sigma: 12.03, eps: {{eps}}, q: 0 }
  - B130: {mw: 776.46, sigma: 11.40, eps: {{eps}}, q: 0 }
  - B131: {mw: 712.42, sigma: 11.08, eps: {{eps}}, q: 0 }
  - B132: {mw: 770.47, sigma: 11.37, eps: {{eps}}, q: 0 }
  - B133: {mw: 748.45, sigma: 11.26, eps: {{eps}}, q: 0 }
  - B134: {mw: 712.41, sigma: 11.08, eps: {{eps}}, q: 0 }
  - B135: {mw: 812.51, sigma: 11.58, eps: {{eps}}, q: 0 }
  - B136: {mw: 820.47, sigma: 11.62, eps: {{eps}}, q: 0 }
  - B137: {mw: 870.51, sigma: 11.85, eps: {{eps}}, q: 0 }
  - B138: {mw: 782.55, sigma: 11.43, eps: {{eps}}, q: 0 }
  - B139: {mw: 704.50, sigma: 11.04, eps: {{eps}}, q: 0 }
  - B140: {mw: 784.52, sigma: 11.44, eps: {{eps}}, q: 0 }
  - B141: {mw: 802.54, sigma: 11.53, eps: {{eps}}, q: 0 }
  - B142: {mw: 818.48, sigma: 11.61, eps: {{eps}}, q: 0 }
  - B143: {mw: 716.47, sigma: 11.10, eps: {{eps}}, q: 0 }
  - B144: {mw: 846.48, sigma: 11.74, eps: {{eps}}, q: 0 }
  - B145: {mw: 886.55, sigma: 11.92, eps: {{eps}}, q: 0 }
  - B146: {mw: 786.48, sigma: 11.45, eps: {{eps}}, q: 0 }
  - B147: {mw: 940.53, sigma: 12.16, eps: {{eps}}, q: 0 }
  - B148: {mw: 782.49, sigma: 11.43, eps: {{eps}}, q: 0 }
  - B149: {mw: 848.53, sigma: 11.75, eps: {{eps}}, q: 0 }
  - B150: {mw: 844.56, sigma: 11.73, eps: {{eps}}, q: 0 }
  - B151: {mw: 736.44, sigma: 11.20, eps: {{eps}}, q: 0 }
  - B152: {mw: 764.45, sigma: 11.34, eps: {{eps}}, q: 0 }
  - B153: {mw: 830.49, sigma: 11.66, eps: {{eps}}, q: 0 }
  - B154: {mw: 854.51, sigma: 11.77, eps: {{eps}}, q: 0 }
  - B155: {mw: 854.53, sigma: 11.77, eps: {{eps}}, q: 0 }
  - B156: {mw: 794.56, sigma: 11.49, eps: {{eps}}, q: 0 }
  - B157: {mw: 816.48, sigma: 11.60, eps: {{eps}}, q: 0 }
  - B158: {mw: 820.47, sigma: 11.62, eps: {{eps}}, q: 0 }
  - B159: {mw: 854.51, sigma: 11.77, eps: {{eps}}, q: 0 }
  - B160: {mw: 768.43, sigma: 11.36, eps: {{eps}}, q: 0 }
  - B161: {mw: 906.59, sigma: 12.01, eps: {{eps}}, q: 0 }
  - B162: {mw: 844.49, sigma: 11.73, eps: {{eps}}, q: 0 }
  - B163: {mw: 810.58, sigma: 11.57, eps: {{eps}}, q: 0 }
  - B164: {mw: 942.58, sigma: 12.16, eps: {{eps}}, q: 0 }
  - B165: {mw: 780.45, sigma: 11.42, eps: {{eps}}, q: 0 }
  # Beads 166–174: smaller terminal/linker beads
  - B166: {mw: 380.18, sigma:  8.99, eps: {{eps}}, q: 0 }
  - B167: {mw: 304.12, sigma:  8.34, eps: {{eps}}, q: 0 }
  - B168: {mw: 342.15, sigma:  8.68, eps: {{eps}}, q: 0 }
  - B169: {mw: 380.18, sigma:  8.99, eps: {{eps}}, q: 0 }
  - B170: {mw: 342.15, sigma:  8.68, eps: {{eps}}, q: 0 }
  - B171: {mw: 342.15, sigma:  8.68, eps: {{eps}}, q: 0 }
  - B172: {mw: 288.12, sigma:  8.19, eps: {{eps}}, q: 0 }
  - B173: {mw: 320.12, sigma:  8.49, eps: {{eps}}, q: 0 }
  - B174: {mw: 320.12, sigma:  8.49, eps: {{eps}}, q: 0 }

moleculelist:
    - mAb:
        structure: {{path_monomer}}
        keepcharges: True
        keeppos: False
        rigid: False

insertmolecules:
    - mAb: {N: {{n}}, inactive: False}

moves:
    - moltransrot: {molecule: mAb, dir: [1,1,1], dprot: 0.1, dp: {{ L/2 }} }

analysis:
    - sanity:           {nstep: {{nstep}} }
    - systemenergy:     {file: energy.dat,  nstep: {{nstep}}, nskip: {{nskip}} }
    - molecule_density: {nstep: {{nstep}} }
    - savestate:        {file: state.json}
    - savestate:        {file: confout.pqr}
    - xtcfile:          {file: traj.xtc,    nstep: {{nstep}}, nskip: {{nskip}} }
    - scatter:          {file: debye.dat,   nstep: 1,         nskip: {{nskip}},
                         molecules: ["mAb"], com: False, scheme: explicit, pmax: 20 }
""")

## 5. Simulation Parameters

Edit the values in this cell to change the conditions explored.

In [ ]:
# --- Physical / chemical conditions ---
pH_range    = [8.55]            # pH values to simulate
I_range     = [0.01, 0.0225, 0.150]  # Ionic strength values [Molar]
cp_range    = [45]              # mAb concentration [mg/mL]
Mw          = 150_000           # mAb molecular weight [g/mol]
I_buffer    = 0.0012            # Buffer ionic strength (10 mM His, pH 6.5) [Molar]

# --- Simulation box / population ---
N_range     = [100]              # Number of mAb molecules per simulation box
replica     = range(3)          # Replica indices (increase for statistics)

# --- Potential parameters ---
eps_values  = [0.3185]           # LJ epsilon values [kJ/mol · kT⁻¹ via to_kJmol]
to_kJmol    = 2.476             # Conversion factor: kT → kJ/mol at 298.15 K

# --- MC step counts ---
micro  = 10       # Micro-steps per macro-step
macro  = 1000    # Number of macro-steps
nstep  = 1000    # Analysis output frequency
nskip  = 0       # Equilibration steps to discard before analysis

print("Simulation conditions:")
print(f"  pH:              {pH_range}")
print(f"  Ionic strength:  {I_range} Molar")
print(f"  Concentration:   {cp_range} mg/mL")
print(f"  N (mAbs / box): {N_range}")
print(f"  Replicas:        {list(replica)}")
print(f"  eps (LJ):        {eps_values} kJ/mol")
print(f"  MC steps:        {macro} macro × {micro} micro")

## 6. Run Monte Carlo Simulations (Yukawa + Lennard-Jones)

For each combination of conditions the cell:
1. Creates the directory tree `I_q/Yukawa_LJ/<N>/<cp>/<pH>/<I>/<eps>/<replica>/`
2. Writes a Faunus `input.yml` via the Jinja2 template
3. Calls Faunus (`yason.py input.yml | faunus`), resuming from `state.json` if it exists

> **Note:** Update `YASON_PATH` and `FAUNUS_PATH` below to match your installation.

In [ ]:
# --- Paths to Faunus executables (edit as needed) ---
YASON_PATH  = "/opt/anaconda3/bin/yason.py"
FAUNUS_PATH = "/opt/anaconda3/bin/faunus"

# --- Restore working directory ---
try:
    workdir
except NameError:
    workdir = %pwd
else:
    %cd -q $workdir

# --- Create output directory ---
%mkdir -p I_q/Yukawa_LJ
%cd I_q/Yukawa_LJ

for n in N_range:
    %mkdir -p $n
    %cd -q $n
    for cp in cp_range:
        %mkdir -p $cp
        %cd -q $cp
        for p in pH_range:
            %mkdir -p $p
            %cd -q $p
            for i in I_range:
                %mkdir -p $i
                %cd -q $i
                for e, eps in enumerate(eps_values):
                    eps = round(eps, 4)
                    %mkdir -p $eps
                    %cd -q $eps
                    for r in replica:
                        %mkdir -p $r
                        %cd -q $r

                        # Resolve input files
                        path_Q       = workdir + '/Inputs/mAb_net_charge.cvs'
                        path_monomer = (workdir + '/Inputs/'
                                        f'mAb1_174-bead_{i}_mM_pH_{p}.pqr')

                        # Compute derived quantities
                        L  = length_in_gl(cp, n, Mw)
                        Q  = read_charge(path_Q, p, i)
                        Dl = debye_length(Mw, cp, Q, i, I_buffer)

                        # Write Faunus input
                        create_input(path_monomer, macro, micro, nstep, nskip,
                                     L, n, Dl, e, eps, i)

                        # Run Faunus (resume from checkpoint if available)
                        if os.path.isfile('state.json'):
                            !export OMP_NUM_THREADS=1; {YASON_PATH} input.yml | {FAUNUS_PATH} --state state.json -v 2
                        else:
                            !export OMP_NUM_THREADS=1; {YASON_PATH} input.yml | {FAUNUS_PATH} -v 2

                        %cd -q ../
                    %cd -q ../
                %cd -q ../
            %cd -q ../
        %cd -q ../
    %cd -q ../

%cd -q ../../
print("Simulations complete.")

## 7. Load Experimental SAXS Data

Three SAXS profiles at $c_p = 45$ mg/mL, pH 8.55, varying ionic strength.

In [ ]:
# --- Restore working directory ---
try:
    workdir
except NameError:
    workdir = %pwd
else:
    %cd -q $workdir

path_exp = workdir + '/Experimental_data'

q_exp_10mM,   I_exp_10mM   = np.loadtxt(path_exp + '/45_10_mAb1_saxs.dat',
                                          unpack=True, usecols=(0, 1), skiprows=12)
q_exp_22p5mM, I_exp_22p5mM = np.loadtxt(path_exp + '/45_22p5_mAb1_saxs.dat',
                                          unpack=True, usecols=(0, 1), skiprows=12)
q_exp_150mM,  I_exp_150mM  = np.loadtxt(path_exp + '/45_150_mAb1_saxs.dat',
                                          unpack=True, usecols=(0, 1), skiprows=12)

print(f"Loaded SAXS data:")
print(f"  IS = 10 mM   : {len(q_exp_10mM)} points, q in [{q_exp_10mM.min():.4f}, {q_exp_10mM.max():.4f}] Å⁻¹")
print(f"  IS = 22.5 mM : {len(q_exp_22p5mM)} points, q in [{q_exp_22p5mM.min():.4f}, {q_exp_22p5mM.max():.4f}] Å⁻¹")
print(f"  IS = 150 mM  : {len(q_exp_150mM)} points, q in [{q_exp_150mM.min():.4f}, {q_exp_150mM.max():.4f}] Å⁻¹")

## 8. Plot $I(Q)$: Simulation vs. Experiment

Simulated Debye scattering profiles (`debye.dat`) are scaled by a single factor `s` and overlaid on the experimental SAXS curves. Individual replicas are shown as semi-transparent points; their mean is shown as a filled marker.

In [ ]:
# --- Restore working directory ---
try:
    workdir
except NameError:
    workdir = %pwd
else:
    %cd -q $workdir

# --- Plot conditions (must match the simulation parameters above) ---
potentials  = ['Yukawa_LJ']
N_plot      = 100
cp_plot     = [45]
pH_plot     = 8.55
IS_plot     = [0.010, 0.0225, 0.150]
eps_plot    = [0.3185]

# --- Scaling factor to match simulation intensities to experiment ---
s = 0.0078

# --- Plot styling ---
plt.rcParams.update({
    'font.size': 16,
    'figure.figsize': [6,6],
    'xtick.major.size': 7,
    'ytick.major.size': 7,
    'legend.frameon': True,
    'legend.labelspacing': 1,
})
colors    = ['palevioletred', 'rebeccapurple', 'darkseagreen',
             'orange', 'purple', 'brown', 'olive', 'k']
linewidth = 1.4

fig, ax = plt.subplots()

# --- Experimental SAXS profiles ---
ax.plot(q_exp_10mM,   I_exp_10mM,   '-', color=colors[0], alpha=0.3,
        lw=linewidth, label='IS = 10 mM — SAXS')
ax.plot(q_exp_22p5mM, I_exp_22p5mM, '-', color=colors[1], alpha=0.3,
        lw=linewidth, label='IS = 22.5 mM — SAXS')
ax.plot(q_exp_150mM,  I_exp_150mM,  '-', color=colors[2], alpha=0.3,
        lw=linewidth, label='IS = 150 mM — SAXS')

# --- Simulated profiles ---
for potential in potentials:
    for C, c in enumerate(cp_plot):
        for I, i in enumerate(IS_plot):
            avg          = []
            q_valid      = None

            for e, eps in enumerate(eps_plot):
                for r in replica:
                    path_iq = (f"{workdir}/I_q/{potential}/{N_plot}/{c}"
                               f"/{pH_plot}/{i}/{eps}/{r}/debye.dat")
                    if os.path.exists(path_iq):
                        q_sim, I_sim = np.loadtxt(path_iq, unpack=True, usecols=(0, 1))
                        avg.append(I_sim * s)
                        q_valid = q_sim
                        ax.plot(q_sim, I_sim * s, '.', color=colors[I],
                                markersize=8, fillstyle='none', alpha=0.2)
                    else:
                        print(f"[WARNING] File not found: {path_iq}")

            if avg and q_valid is not None:
                I_avg = np.mean(avg, axis=0)
                ax.plot(q_valid, I_avg, 'X', color=colors[I], markersize=6,
                        lw=linewidth, alpha=1.0,
                        label=f'IS = {i*1000:.0f} mM — MC')
                # Optional: interpolated line (uncomment to enable)
                # f_interp = interpolate.interp1d(q_valid, I_avg,
                #                                 fill_value='extrapolate', kind='linear')
                # q_dense  = np.arange(0.004, q_valid.min(), 0.0001)
                # ax.plot(q_dense, f_interp(q_dense), '--', color=colors[I], lw=linewidth)
            else:
                print(f"[WARNING] No valid files for c={c}, IS={i}, eps={eps}. Skipping.")

# --- Axes formatting ---
ax.set_xlim([0.004, 0.3])
ax.set_xscale('log')
ax.set_xlabel(r"$q$ [Å$^{-1}$]", fontsize=25)
ax.set_ylabel(r"Scattering Intensity, $I(q)$", fontsize=25)
ax.legend(fontsize=13, ncol=1, frameon=True, loc='upper right')

plt.tight_layout()
plt.show()
